# Sheet 0.2: Linear algebra

**Roadmap topic 0.2.** Read the resources listed on the dashboard first (3Blue1Brown *Essence of Linear Algebra*, MML ch. 2-4, Strang 18.06).

**Rules**
- Fill in every cell marked `# YOUR CODE`. Do not edit cells marked `CHECK`.
- A check prints `PASS` or fails with an `AssertionError` that tells you what is wrong.
- You may use `torch.linalg.qr` and `torch.linalg.svd` / `eigh` only where a task says so. Everything else is yours.
- Answer the **Written** questions in the markdown cells. They are graded by you against the hints at the end, so be honest.

Runtime: CPU is fine for all checks. Time: 4-5 days including reading.

In [ ]:
import math, time, torch
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

## Exercise 1: Top-k eigenpairs by power iteration with deflation

PCA, sharpness estimation and spectral norm all reduce to "find the top eigenvectors of a symmetric PSD matrix without a full decomposition".

Implement:
- `power_iteration(matvec, n, iters, tol)` that returns `(lam, v)` for the largest eigenvalue of a symmetric PSD operator given only `matvec(v)`. Stop when the eigenvalue changes by less than `tol` relatively.
- `top_k_eigs(A, k)` that finds the top-`k` eigenpairs by **deflation**: after finding `(lam, v)`, continue on `A - lam v vᵀ`. Do not build the deflated matrix explicitly; wrap `matvec` instead.

Return eigenvalues sorted descending and eigenvectors as columns of an `(n, k)` matrix.

In [ ]:
def power_iteration(matvec, n, iters=5000, tol=1e-14):
    # YOUR CODE
    raise NotImplementedError

def top_k_eigs(A, k, iters=5000):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    n = 400
    Q, _ = torch.linalg.qr(torch.randn(n, n))
    spec = torch.cat([torch.tensor([50., 30., 20., 12., 8.]), torch.rand(n - 5) * 4])
    A = (Q * spec) @ Q.T
    lam, V = top_k_eigs(A, 5)
    ref = torch.linalg.eigvalsh(A).flip(0)[:5]
    assert lam.shape == (5,) and V.shape == (n, 5), "wrong shapes"
    assert torch.allclose(lam, ref, rtol=1e-8), f"eigenvalues off: {lam} vs {ref}"
    resid = (A @ V - V * lam).norm(dim=0) / lam
    assert (resid < 1e-6).all(), f"eigenvector residuals too big: {resid}"
    print("PASS  top-5:", [round(x, 6) for x in lam.tolist()])
_check1()

**Written (1a):** How many iterations does power iteration need to reach relative accuracy ε, as a function of λ₂/λ₁? What happens on a matrix whose top two eigenvalues are 10.0 and 9.99?

**Written (1b):** Deflation accumulates error: each later eigenpair is computed on a slightly wrong operator. Name a method that avoids this (hint: it iterates on a block of vectors at once) and why it is what libraries actually use.

**Your answer:**

## Exercise 2: Low-rank approximation and Eckart-Young

`low_rank(W, k)` returns the best rank-`k` approximation of `W` in Frobenius norm (use `torch.linalg.svd`).
`predicted_error(S, k)` returns the Frobenius error you expect **from the singular values alone**.
`params_saved(m, n, k)` returns the fraction of parameters saved by storing the factors `U_k diag(S_k)` and `V_kᵀ` instead of `W`.

Then prove Eckart-Young empirically: no other rank-`k` matrix you can think of does better. The check tries random rank-`k` projections against yours.

In [ ]:
def low_rank(W, k):
    # YOUR CODE
    raise NotImplementedError

def predicted_error(S, k):
    # YOUR CODE
    raise NotImplementedError

def params_saved(m, n, k):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    m, n = 300, 200
    U, _ = torch.linalg.qr(torch.randn(m, n)); V, _ = torch.linalg.qr(torch.randn(n, n))
    S = 1.0 / torch.arange(1, n + 1) ** 0.8
    W = (U * S) @ V.T
    for k in (1, 10, 50, 150):
        Wk = low_rank(W, k)
        assert torch.linalg.matrix_rank(Wk) == k, "low_rank does not return rank k"
        err = float((W - Wk).norm())
        assert abs(err - predicted_error(S, k)) < 1e-9, f"k={k}: measured {err}, predicted {predicted_error(S,k)}"
        for _ in range(20):
            P, _ = torch.linalg.qr(torch.randn(m, k))
            assert float((W - P @ (P.T @ W)).norm()) >= err - 1e-12
    assert abs(params_saved(4096, 4096, 64) - (1 - 64 * 8192 / 4096**2)) < 1e-12
    print("PASS")
_check2()

### On Colab: real weights

Run the cell below to repeat the experiment on GPT-2's token-embedding matrix (50257 × 768). Plot the singular value spectrum on a log scale and the relative error `predicted_error(S,k)/‖W‖` for k in 16, 64, 128, 256, 512.

In [ ]:
# Colab only (needs internet). pip install transformers if missing.
from transformers import GPT2Model
W = GPT2Model.from_pretrained("gpt2").wte.weight.detach().double()
S = torch.linalg.svdvals(W)
# YOUR CODE: plot S (log scale) and the relative error curve

**Written (2a):** From your GPT-2 spectrum, what rank keeps 90% of the Frobenius energy? Is the embedding matrix "low rank" in a useful sense?

**Written (2b):** LoRA learns `ΔW = BA` with rank `r`. Using Eckart-Young, explain why a LoRA update can never exactly match a full fine-tuning update whose `ΔW` has many large singular values, and what that predicts about which tasks LoRA struggles with.

**Your answer:**

## Exercise 3: Conditioning, or why you never form XᵀX

1. `make_matrix(m, n, kappa)` builds an `m×n` matrix with **exactly** condition number `kappa` (singular values log-spaced from 1 to 1/kappa).
2. `solve_normal(X, y)` solves least squares through the normal equations `XᵀX w = Xᵀy` (use `torch.linalg.solve`).
3. `solve_qr(X, y)` solves it through a QR factorisation (`torch.linalg.qr` allowed; use a triangular solve).

The check runs both in **float32** for several κ and measures relative error against the true `w`.

In [ ]:
def make_matrix(m, n, kappa, dtype=torch.float64):
    # YOUR CODE
    raise NotImplementedError

def solve_normal(X, y):
    # YOUR CODE
    raise NotImplementedError

def solve_qr(X, y):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    m, n = 400, 50
    for kappa in (1e1, 1e3):
        X = make_matrix(m, n, kappa)
        assert abs(torch.linalg.cond(X) / kappa - 1) < 1e-6, "make_matrix: wrong condition number"
    rows = []
    for kappa in (1e1, 1e2, 1e3):
        X = make_matrix(m, n, kappa); w = torch.randn(n); y = X @ w
        X32, y32 = X.float(), y.float()
        e_n = float((solve_normal(X32, y32).double() - w).norm() / w.norm())
        e_q = float((solve_qr(X32, y32).double() - w).norm() / w.norm())
        rows.append((kappa, e_n, e_q))
    eps = torch.finfo(torch.float32).eps
    for kappa, e_n, e_q in rows:
        assert e_q < 50 * kappa * eps, f"QR error {e_q:.1e} too large at kappa={kappa:.0e}"
    assert rows[-1][1] > 10 * rows[-1][2], "normal equations should be much worse than QR at kappa=1e3"
    for r in rows: print(f"kappa={r[0]:.0e}  normal-eq err={r[1]:.1e}  QR err={r[2]:.1e}")
    print("PASS")
_check3()

**Written (3a):** Your table shows normal-equation error growing roughly like κ² and QR like κ. Explain why in two or three sentences using `cond(XᵀX) = cond(X)²`.

**Written (3b):** Where does this exact issue show up in deep learning? (Think: second-order optimisers like Shampoo/K-FAC inverting `GᵀG`, or Gram matrices in fp16.)

**Your answer:**

## Exercise 4: Randomized SVD

Implement the Halko-Martinsson-Tropp randomized SVD: sketch with a Gaussian test matrix of `k + p` columns, apply `q` power iterations (re-orthonormalise with QR each time), project, then take a small exact SVD. You may use `torch.linalg.qr` and `torch.linalg.svd` on the **small** matrix only.

Target: the top-50 singular values of a 2000×2000 matrix within 1% relative error, and at least 2× faster than a full `torch.linalg.svdvals` on CPU (on a Colab GPU the gap is bigger).

In [ ]:
def randomized_svd(A, k, p=10, q=2):
    # returns U (m,k), S (k,), Vh (k,n)
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    n = 2000
    U, _ = torch.linalg.qr(torch.randn(n, n)); V, _ = torch.linalg.qr(torch.randn(n, n))
    S = torch.exp(-torch.arange(n) / 25.0)
    A = (U * S) @ V.T
    t = time.perf_counter(); ref = torch.linalg.svdvals(A); t_full = time.perf_counter() - t
    t = time.perf_counter(); Uk, Sk, Vhk = randomized_svd(A, 50); t_rand = time.perf_counter() - t
    rel = ((Sk - ref[:50]).abs() / ref[:50]).max()
    assert rel < 0.01, f"max relative error {rel:.3%} > 1%"
    assert torch.allclose(Uk.T @ Uk, torch.eye(50), atol=1e-8), "U columns not orthonormal"
    assert t_full / t_rand > 2, f"only {t_full/t_rand:.1f}x faster than full SVD"
    print(f"PASS  max rel err {rel:.2e}, {t_full/t_rand:.1f}x faster")
_check4()

**Written (4a):** What do the oversampling `p` and power iterations `q` each buy you? When would `q = 0` give poor results?

## Checkpoint (do this without notes)

Explain out loud or in writing:
1. Why LoRA's update `BA` has rank at most `r`.
2. Why the attention score matrix `QKᵀ` (before softmax) has rank at most `d_head`, and what that limits.
3. What a large condition number of a weight matrix tells you about training and about quantisation.

If any answer needs notes, rewatch the matching 3Blue1Brown chapter and Strang lecture before moving to 0.3.

---
### Hints for self-grading written answers
- 1a: error shrinks like (λ₂/λ₁)^t, so t ≈ log ε / log(λ₂/λ₁). With 9.99/10 you need about 23,000 iterations for ε = 1e-10.
- 1b: subspace (block) iteration / Lanczos (LOBPCG).
- 3a: normal equations solve a system with cond κ², and backward-stable QR works on X directly.
- 4a: p absorbs the tail of the spectrum so the top-k subspace is captured; q sharpens slow-decaying spectra (S^(2q+1)). Flat spectra need q > 0.